In [32]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 

from bs4 import BeautifulSoup
import io

import csv

In [ ]:
#### setting dictionaries - mappings 

In [33]:
import os
proj_root = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebooks/ -> project root
data_dir = os.path.join(proj_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)

In [34]:
import os, sys
print(data_dir)
print("cwd:", os.getcwd())
print("src exists:", os.path.exists(os.path.join(os.getcwd(), "src")))
print("sys.path[0]:", sys.path[0])

c:\Users\marki\Data-Processing-in-Python---Project\data\raw
cwd: c:\Users\marki\Data-Processing-in-Python---Project\notebooks
src exists: False
sys.path[0]: c:\Users\marki\AppData\Local\Programs\Python\Python313\python313.zip


In [ ]:
### chmi weather stations - data processing

df_chmi_stat = pd.read_csv(os.path.join(data_dir, "chmi_weather_stations_metadata.csv"))

df_chmi_stat = df_chmi_stat[
    df_chmi_stat["FULL_NAME"].str.contains("Praha", case=False, na=False)
].copy()
wsi_to_drop = [
    "0-203-0-11201020001",  # Praha, Vinohrady - Flora
    "0-203-0-11202007001",  # Praha, Suchdol
    "0-203-0-11105048001",  # Praha, Zadní Kopanina
    "0-203-0-11201020003",  # Praha, Chodov
]
df_chmi_stat = df_chmi_stat[
    ~df_chmi_stat["WSI"].isin(wsi_to_drop)
].copy()
df_chmi_stat["END_YEAR"] = (
    df_chmi_stat["END_DATE"]
    .astype(str)
    .str[:4]
    .astype(int)
)
df_chmi_stat = (
    df_chmi_stat
    .sort_values("END_YEAR", ascending=False)
    .drop_duplicates(subset="WSI", keep="first")
)
current_year = pd.Timestamp.now().year
df_chmi_stat = df_chmi_stat[
    df_chmi_stat["END_YEAR"] >= current_year
].copy()

display(df_chmi_stat)
print(df_chmi_stat.shape)

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_YEAR
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999


(9, 9)


In [38]:
df_chmi_stat

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_YEAR
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999


In [39]:
### discionary of wsi codes

wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

with open(os.path.join(data_dir, "wsi_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(wsi_dict.items())

In [40]:
### chmi weather variables

### filtering only needed ones

df_chmi_vars = pd.read_csv(os.path.join(data_dir, "chmi_weather_variables_metadata.csv"))

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

### dictionary of variables abbreviations and names
chmi_vars_dict = dict(
    zip(
        df_chmi_vars['EG_EL_ABBREVIATION'].astype(str),
        df_chmi_vars['NAME'].astype(str)  
    )
)

with open(os.path.join(data_dir, "chmi_vars_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(chmi_vars_dict.items())

In [41]:
### chmi weather 10min data

df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))

# it gives flag warning, but i assume flag is not of interest


C:\Users\marki\AppData\Local\Temp\ipykernel_16396\1676926942.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))


In [42]:
df_weather.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH
0,0-20000-0-11520,Casmax,2025-01-01T00:00:00Z,381.0,NaN,0.0,0-20000-0-11520,2025,1
1,0-20000-0-11520,Casmax,2025-01-01T00:10:00Z,1.0,NaN,0.0,0-20000-0-11520,2025,1
2,0-20000-0-11520,Casmax,2025-01-01T00:20:00Z,1.0,NaN,0.0,0-20000-0-11520,2025,1
3,0-20000-0-11520,Casmax,2025-01-01T00:30:00Z,201.0,NaN,0.0,0-20000-0-11520,2025,1
4,0-20000-0-11520,Casmax,2025-01-01T00:40:00Z,231.0,NaN,0.0,0-20000-0-11520,2025,1


In [43]:
df_weather['FLAG'].isna().sum()

np.int64(5023823)

In [44]:
df_weather['FLAG'].notna().sum()

np.int64(21577)

In [45]:
df_weather['FLAG'].unique()

array([nan, 'Z', 'N', 'P', 'S'], dtype=object)

In [46]:
df_weather['QUALITY'].unique()

array([0., 4., 3.])

In [47]:
### mapping variables names in weather data 
df_weather['ELEMENT_NAME'] = df_weather['ELEMENT'].map(chmi_vars_dict)

df_weather['WSI_NAME'] = df_weather['WSI'].map(wsi_dict)


In [48]:
df_weather

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH,ELEMENT_NAME,WSI_NAME
0,0-20000-0-11520,Casmax,2025-01-01T00:00:00Z,381.0,NaN,0.0,0-20000-0-11520,2025,1,Čas maxima,"Praha, Libuš"
1,0-20000-0-11520,Casmax,2025-01-01T00:10:00Z,1.0,NaN,0.0,0-20000-0-11520,2025,1,Čas maxima,"Praha, Libuš"
2,0-20000-0-11520,Casmax,2025-01-01T00:20:00Z,1.0,NaN,0.0,0-20000-0-11520,2025,1,Čas maxima,"Praha, Libuš"
3,0-20000-0-11520,Casmax,2025-01-01T00:30:00Z,201.0,NaN,0.0,0-20000-0-11520,2025,1,Čas maxima,"Praha, Libuš"
4,0-20000-0-11520,Casmax,2025-01-01T00:40:00Z,231.0,NaN,0.0,0-20000-0-11520,2025,1,Čas maxima,"Praha, Libuš"
...,...,...,...,...,...,...,...,...,...,...,...
5045395,0-203-0-11201024001,SRA10M,2025-12-31T23:10:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,Srážka-10M,"Praha, Břevnov"
5045396,0-203-0-11201024001,SRA10M,2025-12-31T23:20:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,Srážka-10M,"Praha, Břevnov"
5045397,0-203-0-11201024001,SRA10M,2025-12-31T23:30:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,Srážka-10M,"Praha, Břevnov"
5045398,0-203-0-11201024001,SRA10M,2025-12-31T23:40:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,Srážka-10M,"Praha, Břevnov"


In [49]:
### add metadata about stations

stations_meta = pd.read_csv(os.path.join(data_dir, "chmi_weather_stations_metadata.csv"))

In [50]:
stations_meta

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION
0,0-20000-0-04030,ZIS04030,2015-01-01T00:00:00Z,3999-12-31T23:59:00Z,Reykjavik,-21.903905,64.127653,51.0
1,0-20000-0-11406,L3CHEB01,1863-10-01T00:00:00Z,1919-12-31T23:59:00Z,Cheb,12.362892,50.076212,458.0
2,0-20000-0-11406,L3CHEB01,1933-05-07T00:00:00Z,1938-04-30T23:59:00Z,Cheb,12.388900,50.073900,471.0
3,0-20000-0-11406,L3CHEB01,1943-06-01T00:00:00Z,1945-01-31T23:59:00Z,Cheb,12.388900,50.073900,471.0
4,0-20000-0-11406,L3CHEB01,1951-01-01T00:00:00Z,1960-12-31T23:59:00Z,Cheb,12.388900,50.073900,471.0
...,...,...,...,...,...,...,...,...
5534,0-203-0-42109005003,B4ZITK01,2015-09-16T00:00:00Z,3999-12-31T23:59:00Z,Žítková,17.867778,48.992500,705.0
5535,0-203-0-42109007001,B1PITI01,1944-10-01T00:00:00Z,1961-12-31T23:59:00Z,Pitín,17.890500,49.000900,342.0
5536,0-203-0-42109026001,B1LOPE01,1902-01-01T00:00:00Z,1902-12-31T23:59:00Z,Lopeník,17.792800,48.945800,672.0
5537,0-203-0-42109026001,B1LOPE01,1936-03-15T00:00:00Z,1946-06-30T23:59:00Z,Lopeník,17.792800,48.945800,672.0


In [ ]:
stations_meta = stations_meta[stations_meta["WSI"].isin(wsi_dict)].copy()
stations_meta["END_YEAR"] = stations_meta["END_DATE"].astype(str).str[:4].astype(int)
stations_meta = (
    stations_meta.sort_values("END_YEAR", ascending=False)
    .drop_duplicates(subset="WSI", keep="first")
)
current_year = pd.Timestamp.now().year
df_chmi_stat = stations_meta[
    stations_meta["END_YEAR"] >= current_year
].copy()

display(df_chmi_stat)

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT,END_YEAR
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,NaT,3999
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,NaT,3999
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,NaT,3999
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,NaT,3999
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,NaT,3999
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,NaT,3999
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,NaT,3999
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,NaT,3999
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,NaT,3999


In [53]:
stations_meta

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT,END_YEAR
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,NaT,3999
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,NaT,3999
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,NaT,3999
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,NaT,3999
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,NaT,3999
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,NaT,3999
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,NaT,3999
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,NaT,3999
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,NaT,3999


In [58]:
stations_sel = stations_meta[["WSI", "FULL_NAME", "ELEVATION", "GEOGR1", "GEOGR2"]].copy()
stations_sel = stations_sel.rename(columns={
    "GEOGR1": "LON",
    "GEOGR2": "LAT"
})

df_weather["WSI"] = df_weather["WSI"].astype(str)
stations_sel["WSI"] = stations_sel["WSI"].astype(str)

In [61]:
stations_sel

,WSI,FULL_NAME,ELEVATION,LON,LAT
62,0-20000-0-11518,"Praha, Ruzyně",364.00,14.255556,50.100278
65,0-20000-0-11519,"Praha, Karlov",260.50,14.427778,50.069167
68,0-20000-0-11520,"Praha, Libuš",302.04,14.446944,50.007778
75,0-20000-0-11567,"Praha, Kbely",284.50,14.538056,50.123333
1101,0-203-0-10904013001,"Praha, Komořany",213.00,14.406360,49.988600
1273,0-203-0-11101007001,"Praha, Brdy",862.00,13.818380,49.658890
1429,0-203-0-11201024001,"Praha, Břevnov",355.00,14.352689,50.080843
2214,0-203-0-11514,"Praha, Klementinum",190.70,14.416923,50.086634
2217,0-203-0-11515,"Praha, Klementinum",190.70,14.416436,50.086341


In [55]:
df_weather_ext = df_weather.merge(stations_sel, on="WSI", how="left")


MemoryError: Unable to allocate 269. MiB for an array with shape (7, 5045400) and data type object

In [23]:
df_weather_ext

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH,ELEMENT_NAME,WSI_NAME,FULL_NAME,ELEVATION,LON,LAT
0,0-20000-0-11520,Casmax,2025-01-01T00:00:00Z,381.0,NaN,0.0,0-20000-0-11520,2025,1,NaN,NaN,NaN,NaN,NaN,NaN
1,0-20000-0-11520,Casmax,2025-01-01T00:10:00Z,1.0,NaN,0.0,0-20000-0-11520,2025,1,NaN,NaN,NaN,NaN,NaN,NaN
2,0-20000-0-11520,Casmax,2025-01-01T00:20:00Z,1.0,NaN,0.0,0-20000-0-11520,2025,1,NaN,NaN,NaN,NaN,NaN,NaN
3,0-20000-0-11520,Casmax,2025-01-01T00:30:00Z,201.0,NaN,0.0,0-20000-0-11520,2025,1,NaN,NaN,NaN,NaN,NaN,NaN
4,0-20000-0-11520,Casmax,2025-01-01T00:40:00Z,231.0,NaN,0.0,0-20000-0-11520,2025,1,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045395,0-203-0-11201024001,SRA10M,2025-12-31T23:10:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,NaN,NaN,NaN,NaN,NaN,NaN
5045396,0-203-0-11201024001,SRA10M,2025-12-31T23:20:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,NaN,NaN,NaN,NaN,NaN,NaN
5045397,0-203-0-11201024001,SRA10M,2025-12-31T23:30:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,NaN,NaN,NaN,NaN,NaN,NaN
5045398,0-203-0-11201024001,SRA10M,2025-12-31T23:40:00Z,0.0,NaN,0.0,0-203-0-11201024001,2025,12,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
print((df_weather_ext["FULL_NAME"] != df_weather_ext["WSI_NAME"]).any())
print((df_weather_ext["WSI"] != df_weather_ext["STATION"]).any())

True
False


In [ ]:
### air quality chmi data


[]

In [25]:
air_stations_meta = pd.read_csv(os.path.join(data_dir, "airquality_CHMI_stations_metadata.csv"))

In [26]:
air_stations_meta

,id_registration,station_code,locality_code,locality_name,street,city,lon,lat,alt,component_code,component_name,unit
0,40555,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,SO2,oxid siřičitý,ug/m^3
1,40557,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NO2,oxid dusičitý,ug/m^3
2,40560,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NOx,oxidy dusíku,ug/m^3
3,40559,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,O3,přízemní ozon,ug/m^3
4,40561,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,PM10,částice PM10,ug/m^3
...,...,...,...,...,...,...,...,...,...,...,...,...
479,1648406,TRYCA,TRYC,Rychvald,Mírová,Rychvald,18.377254,49.871670,241 m,INDX,Index kvality ovzduší,NaN
480,1410468,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NO2,oxid dusičitý,ug/m^3
481,1410474,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NOx,oxidy dusíku,ug/m^3
482,1410479,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,PM10,částice PM10,ug/m^3


In [27]:
air_stations_meta = air_stations_meta[air_stations_meta['locality_name'].str.contains('Praha', case=False, na=False)]


In [28]:
air_stations_meta

,id_registration,station_code,locality_code,locality_name,street,city,lon,lat,alt,component_code,component_name,unit
39,783575,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,NO2,oxid dusičitý,ug/m^3
40,783579,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,NOx,oxidy dusíku,ug/m^3
41,783591,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,PM10,částice PM10,ug/m^3
42,1648334,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,INDX,Index kvality ovzduší,NaN
66,41157,ALEGA,ALEG,Praha 2-Legerova (hot spot),Legerov 1843,Praha 2,14.430673,50.072388,219 m,NO2,oxid dusičitý,ug/m^3
...,...,...,...,...,...,...,...,...,...,...,...,...
436,1648433,ABREA,ABRE,Praha 6-Břevnov,Šlikova,Praha 6,14.380116,50.084385,300 m,INDX,Index kvality ovzduší,NaN
480,1410468,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NO2,oxid dusičitý,ug/m^3
481,1410474,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NOx,oxidy dusíku,ug/m^3
482,1410479,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,PM10,částice PM10,ug/m^3


In [29]:
### this is only for one import file, i.e. one day

df_air_qual = pd.read_csv(os.path.join(data_dir, "airquality_CHMI_stations_data.csv"))

In [30]:
df_air_qual

,idRegistration,startTime,idValueType,value
0,10221,2026-05-06T16:00:00Z,8,6.9
1,40237,2026-05-06T16:00:00Z,8,3.5
2,40238,2026-05-06T16:00:00Z,8,74.2
3,40244,2026-05-06T16:00:00Z,8,77.8
4,40257,2026-05-06T16:00:00Z,8,4.8
...,...,...,...,...
467,2181672,2026-05-06T16:00:00Z,8,8.2
468,2181676,2026-05-06T16:00:00Z,8,5.4
469,2187304,2026-05-06T16:00:00Z,8,31.3
470,2187315,2026-05-06T16:00:00Z,8,6.5


In [123]:
### match with metadata based on idregistration

df_air_qual = df_air_qual.rename(columns={
    "idRegistration": "id_registration"
}
)

df_air_qual["id_registration"] = df_air_qual["id_registration"].astype(str)
air_stations_meta["id_registration"] = air_stations_meta["id_registration"].astype(str)

In [124]:
# join

df_air_qual = df_air_qual.merge(air_stations_meta, on="id_registration", how="right")

In [125]:
df_air_qual

,id_registration,startTime,idValueType,value,station_code,street,city,lon,lat,alt,component_code,component_name,unit
0,41157,2026-05-01T12:00:00Z,8,33.3,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,NO2,oxid dusičitý,ug/m^3
1,41156,2026-05-01T12:00:00Z,8,56.6,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,NOx,oxidy dusíku,ug/m^3
2,41158,2026-05-01T12:00:00Z,8,549.0,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,CO,oxid uhelnatý,ug/m^3
3,867419,2026-05-01T12:00:00Z,8,6.4,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,PM2_5,"jemné částice PM2,5",ug/m^3
4,867416,2026-05-01T12:00:00Z,8,14.5,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,PM10,částice PM10,ug/m^3
5,1648574,2026-05-01T12:00:00Z,148,4.0,ALEGA,Legerov 1843,Praha 2,14.430673,50.072388,219 m,INDX,Index kvality ovzduší,NaN
6,40238,2026-05-01T12:00:00Z,8,110.5,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,O3,přízemní ozon,ug/m^3
7,40237,2026-05-01T12:00:00Z,8,3.5,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,PM10,částice PM10,ug/m^3
8,1648379,2026-05-01T12:00:00Z,148,3.0,ASUCA,AV ČR,Praha 6-Suchdol,14.384639,50.126530,277 m,INDX,Index kvality ovzduší,NaN
9,1317399,2026-05-01T12:00:00Z,8,6.5,ABREA,Šlikova,Praha 6,14.380116,50.084385,300 m,NO2,oxid dusičitý,ug/m^3


In [ ]:
### golemio air quality 

#### air quality metadata processing

# station_cols = {
#     'geometry.coordinates': 'coordinates', 
#     'properties.id': 'id', 
#     'properties.name': 'name', 
#     'properties.district': 'district', 
#     'properties.measurement.components.type': 'components'
# }

# air_quality_stations = df[station_cols.keys()]

# air_quality_stations = air_quality_stations.rename(columns=station_cols)

# air_quality_stations = (
#     air_quality_stations.groupby('id', as_index=False)
#     .agg({
#         'coordinates': 'first',
#         'name': 'first',
#         'district': 'first',
#         'components': list
#     })
# )

# air_quality_stations[["lon", "lat"]] = pd.DataFrame(air_quality_stations["coordinates"].tolist(), index=air_quality_stations.index)
# air_quality_stations["lon"] = pd.to_numeric(air_quality_stations["lon"], errors="coerce")
# air_quality_stations["lat"] = pd.to_numeric(air_quality_stations["lat"], errors="coerce")


In [ ]:

#air quality stations dictionary

# air_stat_dict = dict(
#     zip(
#         air_quality_stations['id'].astype(str),
#         air_quality_stations['name'].astype(str)  
#     )
# )

In [ ]:
### loading the dictionaries


def load_wsi_dict(path="data/raw/wsi_dict.csv"):
    df = pd.read_csv(path, dtype=str)
    return dict(df.values)

def load_chmi_vars(path="data/raw/chmi_vars.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)